In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/table-1-(2024-25).csv", nrows=5, skiprows=0)
print(df.columns.tolist())
print(df.head())

['Title', 'HE student enrolments by HE provider, permanent address, level of study, mode of study, entrant marker, sex and academic year', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10']
              Title  \
0          Location   
1    Academic years   
2       Data source   
3   Data collection   
4  Data source link   

  HE student enrolments by HE provider, permanent address, level of study, mode of study, entrant marker, sex and academic year  \
0                                                 UK                                                                              
1                                            2024/25                                                                              
2                                               HESA                                                                              
3                                        051,054,056                   

In [2]:
with open("../data/raw/table-1-(2024-25).csv", encoding="utf-8-sig") as f:
    for i, line in enumerate(f):
        print(i, line.strip()[:150])
        if i >= 30:
            break

0 Title,"HE student enrolments by HE provider, permanent address, level of study, mode of study, entrant marker, sex and academic year",,,,,,,,,
1 Location,UK,,,,,,,,,
2 Academic years,2024/25,,,,,,,,,
3 Data source,HESA,,,,,,,,,
4 Data collection,"051,054,056",,,,,,,,,
5 Data source link,https://www.hesa.ac.uk/data-and-analysis/students/table-1,,,,,,,,,
6 Data file canonical link,https://www.hesa.ac.uk/data-and-analysis/students/table-1.zip,,,,,,,,,
7 Licence,Creative Commons Attribution 4.0 International Licence,,,,,,,,,
8 Code page,Unicode UTF-8,,,,,,,,,
9 Disclaimer,Please note that this data includes rounded totals. Caution must be taken when importing into a pivot table so as not to double count.,,,,,
10 2024/25 total,2863180,,,,,,,,,
11 Last updated,Jan-26,,,,,,,,,
12 ,,,,,,,,,,
13 UKPRN,HE provider,Country of HE provider,Region of HE provider,Entrant marker,Level of study,Mode of study,Academic Year,Category marker,Category,Numb
14 10008071,AA School of Architecture,All,All,All

In [3]:
df = pd.read_csv("../data/raw/table-1-(2024-25).csv", skiprows=13)

print(df.shape)
print()
for col in ["Country of HE provider", "Entrant marker", "Level of study",
            "Mode of study", "Category marker"]:
    print(col, "->", df[col].unique().tolist())
    print()

(811832, 11)

Country of HE provider -> ['All', 'England', 'Scotland', 'Wales', 'Northern Ireland']

Entrant marker -> ['All', 'Entrant', 'Not an entrant']

Level of study -> ['All', 'Postgraduate (research)', 'Postgraduate (taught)', 'All postgraduate', 'First degree', 'Other undergraduate', 'All undergraduate']

Mode of study -> ['All', 'Full-time', 'Part-time']

Category marker -> ['Sex', 'Permanent address', 'Total']



In [4]:
df["Number"] = pd.to_numeric(df["Number"], errors="coerce")

totals = df[
    (df["Country of HE provider"] == "All") &
    (df["Region of HE provider"] == "All") &
    (df["Entrant marker"] == "All") &
    (df["Level of study"] == "All") &
    (df["Mode of study"] == "All") &
    (df["Category marker"] == "Total")
]

print("Rows:", len(totals))
print("Sum:", totals["Number"].sum())
print(totals.sort_values("Number", ascending=False).head(5)[["HE provider", "Number"]])

Rows: 305
Sum: 5726355
                         HE provider   Number
138921                         Total  2863180
95157            The Open University   124580
130801     University College London    51315
80429   The University of Manchester    46305
58645          King's College London    40870


In [5]:
totals = totals[totals["HE provider"] != "Total"]

print("Rows:", len(totals))
print("Sum:", totals["Number"].sum())

Rows: 304
Sum: 2863175


In [6]:
from pathlib import Path

RAW = Path("../data/raw")

def find_header(path):
    """Find the line where the real column headers start."""
    with open(path, encoding="utf-8-sig") as f:
        for i, line in enumerate(f):
            if line.startswith("UKPRN"):
                return i
    raise ValueError(f"No header row found in {path}")

def load_year(path):
    df = pd.read_csv(path, skiprows=find_header(path), dtype=str)
    df = df[
        (df["Country of HE provider"] == "All") &
        (df["Region of HE provider"] == "All") &
        (df["HE provider"] != "Total")
    ].copy()
    df["Category"] = df["Category"].str.strip()
    df["Number"] = pd.to_numeric(df["Number"].str.replace(",", ""), errors="coerce")
    return df

files = sorted(RAW.glob("table-1-*.csv"))
all_years = pd.concat([load_year(p) for p in files], ignore_index=True)
print(all_years.shape)

(1979334, 11)


In [7]:
check = all_years[
    (all_years["Entrant marker"] == "All") &
    (all_years["Level of study"] == "All") &
    (all_years["Mode of study"] == "All") &
    (all_years["Category marker"] == "Total")
]
print(check.groupby("Academic Year")["Number"].agg(["size", "sum"]))

               size      sum
Academic Year               
2014/15         226  2315800
2015/16         261  2332825
2016/17         262  2378020
2017/18         266  2415300
2018/19         266  2457285
2019/20         271  2529850
2020/21         282  2747200
2021/22         285  2857835
2022/23         291  2937260
2023/24         303  2900220
2024/25         304  2863175


In [8]:
published = {}
for p in files:
    with open(p, encoding="utf-8-sig") as f:
        for line in f:
            first = line.split(",")[0]
            if first.endswith(" total"):
                year = first.replace(" total", "")
                published[year] = int(line.split(",")[1].strip('"').replace(",", ""))
                break

comparison = check.groupby("Academic Year")["Number"].sum().to_frame("our_sum")
comparison["published"] = pd.Series(published)
comparison["difference"] = comparison["our_sum"] - comparison["published"]
print(comparison)

               our_sum  published  difference
Academic Year                                
2014/15        2315800    2315840         -40
2015/16        2332825    2332825           0
2016/17        2378020    2378020           0
2017/18        2415300    2415335         -35
2018/19        2457285    2457250          35
2019/20        2529850    2529870         -20
2020/21        2747200    2747200           0
2021/22        2857835    2857855         -20
2022/23        2937260    2937285         -25
2023/24        2900220    2900240         -20
2024/25        2863175    2863180          -5


In [9]:
latest = all_years[all_years["Academic Year"] == "2024/25"]

print("Permanent address categories:")
print(latest[latest["Category marker"] == "Permanent address"]["Category"].unique().tolist())

levels = ["Postgraduate (research)", "Postgraduate (taught)",
          "First degree", "Other undergraduate"]
level_sum = latest[
    (latest["Entrant marker"] == "All") &
    (latest["Mode of study"] == "All") &
    (latest["Category marker"] == "Total") &
    (latest["Level of study"].isin(levels))
]["Number"].sum()

print("\nFour levels added together:", level_sum)
print("Official total:            ", 2863180)

Permanent address categories:
['England', 'Scotland', 'Wales', 'Northern Ireland', 'Other UK', 'Total UK', 'European Union', 'Non-European Union', 'Total Non-UK', 'Not known']

Four levels added together: 2863085
Official total:             2863180


In [10]:
d = all_years.rename(columns={
    "UKPRN": "ukprn", "HE provider": "provider", "Academic Year": "year",
    "Entrant marker": "entrant", "Level of study": "level",
    "Mode of study": "mode", "Category marker": "marker",
    "Category": "category", "Number": "number"
})

def is_all(col):
    return d[col] == "All"

is_total = d["marker"] == "Total"

def section(name, mask, group_col=None):
    out = d[mask].copy()
    out["measure"] = name
    out["group"] = out[group_col] if group_col else "All"
    return out[["ukprn", "year", "measure", "group", "number"]]

domiciles = ["Total UK", "European Union", "Non-European Union", "Not known"]

clean = pd.concat([
    section("total", is_all("entrant") & is_all("level") & is_all("mode") & is_total),
    section("level", is_all("entrant") & is_all("mode") & is_total & d["level"].isin(levels), "level"),
    section("mode", is_all("entrant") & is_all("level") & is_total & d["mode"].isin(["Full-time", "Part-time"]), "mode"),
    section("domicile", is_all("entrant") & is_all("level") & is_all("mode")
            & (d["marker"] == "Permanent address") & d["category"].isin(domiciles), "category"),
    section("entrants", (d["entrant"] == "Entrant") & is_all("level") & is_all("mode") & is_total),
], ignore_index=True)

clean["group"] = clean["group"].replace({"Total UK": "UK"})

# Use each university's most recent name, in case it changed over the years
latest_names = d.sort_values("year").groupby("ukprn")["provider"].last()
clean["provider"] = clean["ukprn"].map(latest_names)

print("Duplicate rows:", clean.duplicated(["ukprn", "year", "measure", "group"]).sum())
print(clean.shape)
print(clean.head(10))

clean.to_csv("../data/processed/enrolments_clean.csv", index=False)

Duplicate rows: 0
(32480, 6)
      ukprn     year measure group  number                           provider
0  10007783  2014/15   total   All   14035         The University of Aberdeen
1  10019746  2014/15   total   All     110                ABI College Limited
2  10007849  2014/15   total   All    4220                 Abertay University
3  10007856  2014/15   total   All    9835             Aberystwyth University
4  10000080  2014/15   total   All      70            Access to Music Limited
5  10000248  2014/15   total   All     195  Academy of Live and Recorded Arts
6  10000291  2014/15   total   All   19830           Anglia Ruskin University
7  10000381  2014/15   total   All     245           Arts Educational Schools
8  10007759  2014/15   total   All   11070                   Aston University
9  10007857  2014/15   total   All   10765                  Bangor University
